# How agents are actually measured [Step 08.01 - tau-bench, GAIA, SWE-bench]

> **MLCourse - Agentic AI - Agent Patterns**

Every model release comes with a table of agent benchmark scores. This notebook
explains what three of the most-cited ones actually measure, so that the numbers
stop being decoration - and so that the harness you build in notebook 02 is shaped
like a real one.

**No datasets are downloaded here.** These benchmarks are large, some are gated, and
running them properly costs real money. The point is the *design*, which you can
learn from a description and then reimplement at 1/1000 scale.

### Key takeaways

- A benchmark is four separable pieces: a **task set**, an **environment**, an
  **agent interface**, and a **grader**. If the grader is inside the agent, it is a
  demo, not a benchmark.
- The grader is the hard part. Every serious agent benchmark spends most of its
  design effort there.
- Agent benchmarks are **stochastic**. A single run is a sample, not a score - which
  is why notebooks 03 and 04 exist.

### 1. tau-bench (τ-bench) - tool use in a real conversation

**What it measures:** whether an agent can hold a customer-service conversation
against a real API and leave the **database in the correct final state**.

**The design.** An agent plays a retail or airline support agent. Another LLM plays
the *user*, with a hidden goal ("change my flight to Tuesday, but only if it costs
under $200"). The agent has real tools that mutate a database, and a written policy
document it must obey.

**The grader is the interesting part.** It does not read the conversation. It
compares the **final database state** against the state a correct agent would have
produced. That is a deterministic check on an open-ended interaction - the single
best idea in agent evaluation.

**Its signature metric: `pass^k`** (pass-to-the-power-k, *not* pass@k). The same
task is run k times and it counts only if the agent succeeds **every** time. This is
a *reliability* metric, and it is brutal by design: an agent with 80% per-run success
scores 0.8^8 ≈ 17% at k=8. That is the right question for customer service, where
one wrong refund is not averaged away by nine right ones.

> Note the contrast with pass@k (notebook 03), which counts a task as solved if
> **any** of k attempts succeeds. pass@k is optimistic (you can retry); pass^k is
> pessimistic (you cannot). They answer different questions and are trivially
> confused because the names look alike.

### 2. GAIA - general assistant tasks

**What it measures:** multi-step real-world questions that are *easy for humans and
hard for models* - "conceptually simple, procedurally long".

**The design.** ~450 questions across three difficulty levels, each needing some mix
of web browsing, file handling (spreadsheets, PDFs, images) and multi-hop reasoning.
Level 1 needs a handful of steps and typically one tool; level 3 can need dozens of
steps and several tools.

**The grader is deliberately trivial:** questions are written so that the answer is a
short string or number, checked by **quasi-exact match** after normalisation. All the
difficulty is pushed into the *question*, none into the *grading*. This is a design
choice worth stealing: **if you can phrase the task so a string comparison grades it,
do that instead of building a judge.**

**Its lesson.** Humans score ~92%. Early GPT-4-with-plugins scored ~15%. The gap is
not knowledge - it is *procedure*: staying on task across many steps without losing
the thread. That is precisely the capability the context-engineering and memory
modules of this track exist to support.

Note the practical trap: GAIA's held-out test set answers are private, so numbers
quoted from the validation split are not comparable to leaderboard numbers.

### 3. SWE-bench - real software engineering

**What it measures:** given a real GitHub issue and the repository at the commit
before the fix, can the agent produce a patch that makes the project's tests pass?

**The design.** ~2,300 issue/PR pairs from 12 popular Python repositories. The agent
sees the issue text and the repo. It does *not* see the tests.

**The grader is execution.** The harness applies the agent's patch in a container and
runs two test sets:

- **FAIL_TO_PASS** - tests that failed before and must now pass. Did you fix it?
- **PASS_TO_PASS** - tests that passed before and must still pass. Did you break
  anything?

This is the strongest grader in the list: no judge, no string matching, no opinion.
It is also the most expensive - every evaluation is a container build and a test run.

**Its lessons.**

- **Contamination is a first-class concern.** These are public repositories, so a
  model may have memorised the fix. SWE-bench Verified (human-filtered) and rolling
  variants like SWE-bench Live exist for this reason. Any benchmark built from public
  data ages.
- **The environment is most of the work.** Reproducible per-repo containers are
  harder to build than the task list.
- Report **resolved rate**, cost per instance and the model version together, or the
  number means nothing.

### 4. What they have in common

| | tau-bench | GAIA | SWE-bench |
|---|---|---|---|
| Task source | hand-written policy scenarios | hand-written questions | real GitHub issues |
| Environment | mutable database + tools | web + files | a repo in a container |
| Grader | **final DB state** | **quasi-exact match** | **the test suite** |
| Headline metric | `pass^k` (reliability) | accuracy | resolved rate |
| Judge involved | no | no | no |

**Not one of them uses an LLM judge for the headline number.** Every one of them
found a way to make the check deterministic - by constraining the answer format, by
inspecting state, or by executing code.

That is the design rule this module is built on, and it is why
[`../07_llm_as_judge`](../07_llm_as_judge) ends by telling you to prefer a checker
over a judge wherever one exists.

### 5. The four pieces, restated

Whatever you build, it needs these, and they must be **separable**:

```
TASK SET      fixed, versioned, small enough to run often
ENVIRONMENT   deterministic; same starting state for every attempt
AGENT         a function: task -> answer. Instrumented for tokens and steps.
GRADER        deterministic; NEVER shares code with the agent
```

Two rules that follow immediately, and that people break constantly:

1. **Reset the environment between attempts.** If task 3 sees state left by task 2,
   your task set has a hidden ordering dependency and your results are not
   reproducible.
2. **The grader must not be able to see the agent's reasoning** - only its final
   answer, or the state it left behind. A grader that reads the reasoning rewards
   plausible-sounding failure.

### Next

Notebook 02 builds exactly that, at a scale you can run in a couple of minutes.